In [10]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve, roc_auc_score

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors

from pathlib import Path

In [ ]:
base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'
evaluated_path = base_artifacts / 'Datasets' / 'Evaluated'

baselines = ["ATE", "STE", "Pearson", "DoC", "ABLT", "ABLT/STD", "Cosine", "Jaccard", "ATE (stabilized)", "STE (stabilized)"]

def load_results(DATASET):
    raw_results = pd.read_csv(evaluated_path / f'{DATASET}_evaluated.csv')

    new_names = {
        'ATE_STABILIZED': 'ATE (stabilized)',
        'cosine_similarity': 'Cosine',
        'correlation': 'Pearson',
        'diff_of_conditionals': 'DoC',
        'jacard_index': 'Jaccard',
    }
    raw_results = raw_results.rename(columns=new_names)

    standardize = lambda row, col1, col2: row[col1] / np.max([row[col2], 1e-8])
    new_cols = {
        "STE": ("ATE", "STD"),
        "ABLT/STD": ("ATE_ABLT", "STD_ABLT"),
        "STE (stabilized)": ("ATE (stabilized)", "STD_STABILIZED")
    }
    for new_col in new_cols:
        col1, col2 = new_cols[new_col]
        raw_results[new_col] = raw_results.apply(standardize, axis=1, args=(col1, col2))

    raw_results.rename(columns={'ATE_ABLT': 'ABLT'}, inplace=True)
    return raw_results

In [12]:
# --------------------------------------------------
# 1) Compute summary exactly as you already do
# --------------------------------------------------
datasets = ['ml-10m', 'steam', 'goodreads']
metrics = ['AP(%)', 'AUC(%)']
gpts = [f'GPT>{GPT}' for GPT in range(3)]

summary = {
    col: {
        (dataset, metric, gpt): None
        for dataset in datasets
        for metric in metrics
        for gpt in gpts
    }
    for col in baselines
}

for DATASET in datasets:
    raw_results = load_results(DATASET)
    for GPT in range(3):
        y_true = (raw_results['causal_effect'] > GPT).astype(int).values
        for col in baselines:
            y_score = raw_results[col].values

            ap = average_precision_score(y_true, y_score)
            auc = roc_auc_score(y_true, y_score)

            summary[col][(DATASET, 'AP(%)',  f'GPT>{GPT}')] = ap
            summary[col][(DATASET, 'AUC(%)', f'GPT>{GPT}')] = auc

summary_df = pd.DataFrame(summary).T

# Keep numeric version for comparisons
summary_num = summary_df.copy()

# --------------------------------------------------
# 2) Configuration to match the attached tex exactly
# --------------------------------------------------
dataset_order = ['goodreads', 'ml-10m', 'steam']
dataset_labels = {
    'goodreads': 'Goodreads',
    'ml-10m': 'ML10M',
    'steam': 'Steam'
}
metric_order = ['AP(%)', 'AUC(%)']
gpt_order = ['GPT>0', 'GPT>1', 'GPT>2']

# Optional: rename methods for display in the table
method_labels = {
    'Cosine': 'Cosine',
    'Pearson': 'Pearson',
    'DoC': 'DoC',
    'Jaccard': 'Jaccard',
    'SASRec': 'SASRec',
    'ABLT': 'ABLT',
    'ABLT/STD': 'ABLT/STD',
    'ATE': 'ATE',
    'STE': 'STE',
}

# If you want a specific row order, define it here.
# Otherwise, it will use the current order in summary_df.index.
method_order = [m for m in baselines if m in summary_num.index]

# --------------------------------------------------
# 3) Precompute which entries should be bolded
#    Best per column = best method for each
#    (dataset, metric, GPT)
# --------------------------------------------------
best_mask = pd.DataFrame(False, index=summary_num.index, columns=summary_num.columns)

for col_key in summary_num.columns:
    col_values = summary_num[col_key].astype(float)
    max_val = col_values.max()
    best_mask[col_key] = col_values.eq(max_val)

# --------------------------------------------------
# 4) Formatter for each cell
# --------------------------------------------------
def fmt_cell(method, dataset, metric, gpt):
    value = summary_num.loc[method, (dataset, metric, gpt)]
    s = f"{100 * value:.2f}"
    if best_mask.loc[method, (dataset, metric, gpt)]:
        return rf"\textbf{{{s}}}"
    return s

# --------------------------------------------------
# 5) Build table body
#    Same layout as attached tex:
#    one row per method, 18 numeric cells total
# --------------------------------------------------
body_lines = []

for method in method_order:
    row_cells = []
    for dataset in dataset_order:
        for metric in metric_order:
            for gpt in gpt_order:
                row_cells.append(fmt_cell(method, dataset, metric, gpt))

    method_name = method_labels.get(method, method)

    line = method_name.ljust(10) + " & " + " & ".join(row_cells) + r" \\"
    body_lines.append(line)

body_str = "\n".join(body_lines)

# --------------------------------------------------
# 6) Assemble full latex table string
#    Caption/label copied from your attached tex.
#    You can edit them freely.
# --------------------------------------------------
latex_table = rf"""
\begin{{table*}}[t]
\centering
\small
\setlength{{\tabcolsep}}{{3.5pt}}
\renewcommand{{\arraystretch}}{{1.05}}
\begin{{tabular}}{{l|ccc ccc|ccc ccc|ccc ccc}}
\toprule
& \multicolumn{{6}}{{c|}}{{\textbf{{Goodreads}}}} 
& \multicolumn{{6}}{{c|}}{{\textbf{{ML10M}}}} 
& \multicolumn{{6}}{{c}}{{\textbf{{Steam}}}} \\
\cmidrule(lr){{2-7}}\cmidrule(lr){{8-13}}\cmidrule(lr){{14-19}}
& \multicolumn{{3}}{{c}}{{AP (\%)}} & \multicolumn{{3}}{{c|}}{{AUC (\%)}} 
& \multicolumn{{3}}{{c}}{{AP (\%)}} & \multicolumn{{3}}{{c|}}{{AUC (\%)}} 
& \multicolumn{{3}}{{c}}{{AP (\%)}} & \multicolumn{{3}}{{c}}{{AUC (\%)}} \\
\cmidrule(lr){{2-4}}\cmidrule(lr){{5-7}}
\cmidrule(lr){{8-10}}\cmidrule(lr){{11-13}}
\cmidrule(lr){{14-16}}\cmidrule(lr){{17-19}}
Method 
& $>$0 & $>$1 & $>$2 & $>$0 & $>$1 & $>$2
& $>$0 & $>$1 & $>$2 & $>$0 & $>$1 & $>$2
& $>$0 & $>$1 & $>$2 & $>$0 & $>$1 & $>$2 \\
\midrule
{body_str}
\bottomrule
\end{{tabular}}
\caption{{Main results on real-world organic CF logs: uncertainty-aware deconfounding improves \emph{{directed}} causal link discovery.
Each method induces a ranking over directed item pairs $(i\!\rightarrow\!j)$ and is evaluated using deterministic \texttt{{gpt-5.2}} ordinal labels \texttt{{causal\_effect}}$\in\{{0,1,2,3\}}$ as an external plausibility signal (Sec.~\ref{{sec:LLM-based-labeling}}).
For each threshold $t\in\{{0,1,2\}}$, we define positives as \texttt{{causal\_effect}}$>t$ (higher $t$ corresponds to more explicit funnel mechanisms, e.g., series/franchise links, and is more selective).
We report AP (primary for rare positives) and ROC-AUC; best per block is \textbf{{bold}}.
$\widehat{{STE}}_{{i\to j}}$ consistently dominates $\widehat{{ATE}}_{{i\to j}}$ and non-causal association baselines, with the largest gains at $t\ge 1$, supporting $\widehat{{STE}}_{{i\to j}}$ as a robust statistic for mining high-confidence causal item influences in sparse, confounded logs.}}
\label{{tab:ap_auc_all_datasets}}
\end{{table*}}
""".strip()

print(latex_table)

\begin{table*}[t]
\centering
\small
\setlength{\tabcolsep}{3.5pt}
\renewcommand{\arraystretch}{1.05}
\begin{tabular}{l|ccc ccc|ccc ccc|ccc ccc}
\toprule
& \multicolumn{6}{c|}{\textbf{Goodreads}} 
& \multicolumn{6}{c|}{\textbf{ML10M}} 
& \multicolumn{6}{c}{\textbf{Steam}} \\
\cmidrule(lr){2-7}\cmidrule(lr){8-13}\cmidrule(lr){14-19}
& \multicolumn{3}{c}{AP (\%)} & \multicolumn{3}{c|}{AUC (\%)} 
& \multicolumn{3}{c}{AP (\%)} & \multicolumn{3}{c|}{AUC (\%)} 
& \multicolumn{3}{c}{AP (\%)} & \multicolumn{3}{c}{AUC (\%)} \\
\cmidrule(lr){2-4}\cmidrule(lr){5-7}
\cmidrule(lr){8-10}\cmidrule(lr){11-13}
\cmidrule(lr){14-16}\cmidrule(lr){17-19}
Method 
& $>$0 & $>$1 & $>$2 & $>$0 & $>$1 & $>$2
& $>$0 & $>$1 & $>$2 & $>$0 & $>$1 & $>$2
& $>$0 & $>$1 & $>$2 & $>$0 & $>$1 & $>$2 \\
\midrule
ATE        & 88.63 & 84.77 & 67.06 & 84.81 & 88.75 & 86.43 & 27.94 & 24.07 & 19.25 & 70.58 & 82.62 & 87.42 & 38.35 & 32.60 & 24.35 & 72.94 & 78.18 & 82.56 \\
STE        & \textbf{89.14} & \textbf{86.14} & \textbf{